# Taller 4 – Registro de Imágenes Médicas
## RegLib Case #20: Intra-subject whole-body PET-CT
**Pontificia Universidad Javeriana** — Procesamiento de Imágenes Médicas, 2026
Integrantes: Abel Albuez Sanchez, Victoria Acero, Santiago Gil

## Paso 1: Instalación de dependencias

In [ ]:
!pip install itk matplotlib numpy -q

## Paso 2: Descarga de imágenes desde GitHub

In [ ]:
# Descarga de las 4 imágenes NRRD desde el repositorio de GitHub
import os
import urllib.request

# URL base de los archivos "raw" en GitHub
BASE_URL = "https://raw.githubusercontent.com/AbelAlbuez/medical-image-processing/main/RegLib_C20_Data/images/"
archivos = ["CT_1.nrrd", "CT_2.nrrd", "PET_1.nrrd", "PET_2.nrrd"]

# Carpeta local de destino dentro del entorno de Colab
IMAGES_DIR = "/content/images"
os.makedirs(IMAGES_DIR, exist_ok=True)

# Descargar cada archivo e informar su tamaño en MB
for nombre in archivos:
    url = BASE_URL + nombre
    destino = os.path.join(IMAGES_DIR, nombre)
    urllib.request.urlretrieve(url, destino)
    tam_mb = os.path.getsize(destino) / (1024 * 1024)
    print(f"Descargado: {nombre}  ({tam_mb:.2f} MB)")

print("\nTodas las imágenes fueron descargadas en", IMAGES_DIR)

## Paso 3: Carga y exploración de imágenes

In [ ]:
# Importación de bibliotecas principales
import itk
import numpy as np
import matplotlib.pyplot as plt
import time

# Marca de tiempo global para medir la duración total del pipeline CT
tiempo_inicio_total = time.time()

# Tipos ITK del proyecto: pixel float32 (itk.F) en 3 dimensiones
PixelType = itk.F
Dimension = 3
ImageType = itk.Image[PixelType, Dimension]


def read_image(path):
    # Lee una imagen NRRD y la devuelve como itk.Image[F, 3]
    reader = itk.ImageFileReader[ImageType].New()
    reader.SetFileName(path)
    reader.Update()
    return reader.GetOutput()


def print_image_info(name, image):
    # Imprime la metadata principal de una imagen ITK
    size = image.GetLargestPossibleRegion().GetSize()
    spacing = image.GetSpacing()
    origin = image.GetOrigin()
    print(f"{name}:")
    print(f"  size    = [{size[0]}, {size[1]}, {size[2]}]  (vóxeles)")
    print(f"  spacing = [{spacing[0]:.4f}, {spacing[1]:.4f}, {spacing[2]:.4f}]  (mm)")
    print(f"  origin  = [{origin[0]:.4f}, {origin[1]:.4f}, {origin[2]:.4f}]")


# Lectura de las cuatro imágenes
ct1 = read_image("/content/images/CT_1.nrrd")    # CT fijo (referencia)
ct2 = read_image("/content/images/CT_2.nrrd")    # CT móvil (a registrar)
pet1 = read_image("/content/images/PET_1.nrrd")  # PET fijo (referencia)
pet2 = read_image("/content/images/PET_2.nrrd")  # PET móvil (a registrar)

# Exploración de la metadata de cada imagen
print_image_info("CT_1", ct1)
print_image_info("CT_2", ct2)
print_image_info("PET_1", pet1)
print_image_info("PET_2", pet2)

## Paso 4: Registro CT — Etapa 1 (Rígido)
Descripción breve: corrige traslaciones y rotaciones globales entre CT_1 y CT_2.
Elementos: Euler3DTransform, MattesMutualInformation, RegularStepGradientDescent, LinearInterpolator.

> **Nota de implementación:** se emplea `VersorRigid3DTransform` (equivalente rígido en 3D) en lugar de `Euler3DTransform`, ya que ITK Python no expone `CenteredTransformInitializer` para Euler3D. Ambas representan rotación + traslación; VersorRigid parametriza la rotación con un versor (cuaternión unitario).

In [ ]:
# Etapa 1 - Registro Rígido
# Nota: se usa VersorRigid3DTransform en lugar de Euler3DTransform porque ITK
# Python no envuelve CenteredTransformInitializer para Euler3D. Ambas son
# rígidas en 3D (rotación + traslación).
fixed = ct1   # imagen fija
moving = ct2  # imagen móvil

tiempo_inicio_etapa = time.time()  # inicio de medición de la etapa

# Transformación rígida e inicialización por momentos (alinea centros de masa)
RigidType = itk.VersorRigid3DTransform[itk.D]
rigid_transform = RigidType.New()

InitializerType = itk.CenteredTransformInitializer[RigidType, ImageType, ImageType]
initializer = InitializerType.New()
initializer.SetTransform(rigid_transform)
initializer.SetFixedImage(fixed)
initializer.SetMovingImage(moving)
initializer.MomentsOn()
initializer.InitializeTransform()

# Métrica: Información Mutua de Mattes (50 bins de histograma)
MetricType = itk.MattesMutualInformationImageToImageMetricv4[ImageType, ImageType]
metric = MetricType.New()
metric.SetNumberOfHistogramBins(50)

# Optimizador: Regular Step Gradient Descent
OptimizerType = itk.RegularStepGradientDescentOptimizerv4[itk.D]
optimizer = OptimizerType.New()
optimizer.SetLearningRate(1.0)
optimizer.SetMinimumStepLength(0.001)
optimizer.SetNumberOfIterations(200)
optimizer.SetRelaxationFactor(0.5)

# Método de registro v4 (interpolador lineal por defecto)
RegistrationType = itk.ImageRegistrationMethodv4[ImageType, ImageType]
registration = RegistrationType.New()
registration.SetFixedImage(fixed)
registration.SetMovingImage(moving)
registration.SetMetric(metric)
registration.SetOptimizer(optimizer)
registration.SetInitialTransform(rigid_transform)
registration.InPlaceOn()

# Esquema multi-resolución de un solo nivel
registration.SetNumberOfLevels(1)
registration.SetSmoothingSigmasPerLevel([0])
registration.SetShrinkFactorsPerLevel([1])

registration.Update()

print(f"  Iteraciones: {optimizer.GetCurrentIteration()}")
print(f"  Métrica final: {optimizer.GetValue():.6f}")
tiempo_etapa = time.time() - tiempo_inicio_etapa
print(f"[Etapa 1] Rígido completada ✓ — Tiempo: {tiempo_etapa:.1f}s ({tiempo_etapa/60:.1f} min)")

## Paso 5: Registro CT — Etapa 2 (Affine)
Descripción breve: agrega escala y cizallamiento sobre el resultado rígido.
Elementos: AffineTransform, MattesMutualInformation, RegularStepGradientDescent.

In [ ]:
# Etapa 2 - Registro Affine, inicializado desde la transform rígida
tiempo_inicio_etapa = time.time()  # inicio de medición de la etapa

AffineType = itk.AffineTransform[itk.D, Dimension]
affine_transform = AffineType.New()

# Copiar centro, matriz y traslación del resultado rígido como punto de partida
affine_transform.SetCenter(rigid_transform.GetCenter())
affine_transform.SetMatrix(rigid_transform.GetMatrix())
affine_transform.SetTranslation(rigid_transform.GetTranslation())

# Misma métrica (Mattes MI, 50 bins)
MetricType = itk.MattesMutualInformationImageToImageMetricv4[ImageType, ImageType]
metric = MetricType.New()
metric.SetNumberOfHistogramBins(50)

# Optimizador con tasa de aprendizaje y paso mínimo más finos
OptimizerType = itk.RegularStepGradientDescentOptimizerv4[itk.D]
optimizer = OptimizerType.New()
optimizer.SetLearningRate(0.1)
optimizer.SetMinimumStepLength(0.0001)
optimizer.SetNumberOfIterations(200)
optimizer.SetRelaxationFactor(0.5)

RegistrationType = itk.ImageRegistrationMethodv4[ImageType, ImageType]
registration = RegistrationType.New()
registration.SetFixedImage(fixed)
registration.SetMovingImage(moving)
registration.SetMetric(metric)
registration.SetOptimizer(optimizer)
registration.SetInitialTransform(affine_transform)
registration.InPlaceOn()

registration.SetNumberOfLevels(1)
registration.SetSmoothingSigmasPerLevel([0])
registration.SetShrinkFactorsPerLevel([1])

registration.Update()

print(f"  Iteraciones: {optimizer.GetCurrentIteration()}")
print(f"  Métrica final: {optimizer.GetValue():.6f}")
tiempo_etapa = time.time() - tiempo_inicio_etapa
print(f"[Etapa 2] Affine completada ✓ — Tiempo: {tiempo_etapa:.1f}s ({tiempo_etapa/60:.1f} min)")

## Paso 6: Registro CT — Etapa 3 (BSpline)
Descripción breve: refinamiento no rígido local para capturar deformaciones de postura.
Elementos: BSplineTransform orden 3, malla 8x8x8, GradientDescentOptimizerv4 (30 it/nivel), pirámide multi-resolución de 3 niveles y muestreo aleatorio (RANDOM) del 5% de la métrica para acotar el tiempo.

In [ ]:
# Etapa 3 - Registro BSpline (refinamiento deformable, orden 3)
tiempo_inicio_etapa = time.time()  # inicio de medición de la etapa

SplineOrder = 3
BSplineType = itk.BSplineTransform[itk.D, Dimension, SplineOrder]
bspline_transform = BSplineType.New()

# Inicializar la malla BSpline (8x8x8) cubriendo el dominio de la imagen fija
InitializerType = itk.BSplineTransformInitializer[BSplineType, ImageType]
bspline_initializer = InitializerType.New()
bspline_initializer.SetTransform(bspline_transform)
bspline_initializer.SetImage(fixed)
mesh = itk.Size[Dimension]()
mesh[0] = 8
mesh[1] = 8
mesh[2] = 8
bspline_initializer.SetTransformDomainMeshSize(mesh)
bspline_initializer.InitializeTransform()

# Métrica: Mattes MI (50 bins)
MetricType = itk.MattesMutualInformationImageToImageMetricv4[ImageType, ImageType]
metric = MetricType.New()
metric.SetNumberOfHistogramBins(50)

# Optimizador: GradientDescent honra bien el budget de iteraciones y el
# esquema multi-resolución. 30 it por nivel para acotar tiempo.
OptimizerType = itk.GradientDescentOptimizerv4Template[itk.D]
optimizer = OptimizerType.New()
optimizer.SetLearningRate(1.0)
optimizer.SetNumberOfIterations(30)
optimizer.SetConvergenceWindowSize(5)

RegistrationType = itk.ImageRegistrationMethodv4[ImageType, ImageType]
registration = RegistrationType.New()
registration.SetFixedImage(fixed)
registration.SetMovingImage(moving)
registration.SetMetric(metric)
registration.SetOptimizer(optimizer)
registration.SetInitialTransform(bspline_transform)
registration.InPlaceOn()

# La affine se aplica como "moving initial transform": se compone con la
# BSpline durante la optimización.
registration.SetMovingInitialTransform(affine_transform)

# Pirámide multi-resolución: 3 niveles, de grueso a fino. Acelera mucho
# respecto a evaluar siempre a resolución completa.
registration.SetNumberOfLevels(3)
registration.SetSmoothingSigmasPerLevel([2, 1, 0])
registration.SetShrinkFactorsPerLevel([4, 2, 1])
registration.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

# Muestreo estocástico de la métrica MI: ~5% de voxels por iteración.
SamplingStrategy = itk.ImageRegistrationMethodv4.RANDOM
registration.SetMetricSamplingStrategy(SamplingStrategy)
registration.SetMetricSamplingPercentage(0.05)

registration.Update()

tiempo_etapa = time.time() - tiempo_inicio_etapa
print(f"[Etapa 3] BSpline completada ✓ — Tiempo: {tiempo_etapa:.1f}s ({tiempo_etapa/60:.1f} min)")

## Paso 7: Resampling CT_2 registrado y guardado de transformaciones

In [ ]:
# Composición de transformaciones: primero affine, luego BSpline
# (orden de aplicación a un punto: T_total(p) = T_affine(T_bspline(p)))
tiempo_inicio_etapa = time.time()  # inicio de medición de la etapa

CompositeType = itk.CompositeTransform[itk.D, Dimension]
ct_composite = CompositeType.New()
ct_composite.AddTransform(affine_transform)
ct_composite.AddTransform(bspline_transform)

# Resampling de CT_2 al espacio de CT_1
ResamplerType = itk.ResampleImageFilter[ImageType, ImageType]
resampler = ResamplerType.New()
resampler.SetInput(moving)               # CT_2
resampler.SetTransform(ct_composite)
resampler.SetUseReferenceImage(True)
resampler.SetReferenceImage(fixed)       # metadata de CT_1
resampler.SetDefaultPixelValue(-1000.0)  # HU del aire

InterpolatorType = itk.LinearInterpolateImageFunction[ImageType, itk.D]
resampler.SetInterpolator(InterpolatorType.New())
resampler.Update()
ct2_registered = resampler.GetOutput()

# Imagen diferencia |CT_1 - CT_2_registered|
sub = itk.SubtractImageFilter[ImageType, ImageType, ImageType].New()
sub.SetInput1(fixed)
sub.SetInput2(ct2_registered)
abs_filter = itk.AbsImageFilter[ImageType, ImageType].New()
abs_filter.SetInput(sub.GetOutput())
abs_filter.Update()
ct_diff = abs_filter.GetOutput()

# Carpetas de salida en el entorno de Colab
RESULTS_DIR = "/content/results"
TRANSFORMS_DIR = "/content/transforms"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(TRANSFORMS_DIR, exist_ok=True)


def write_image(image, path):
    # Escribe una itk.Image[F, 3] en disco
    writer = itk.ImageFileWriter[ImageType].New()
    writer.SetFileName(path)
    writer.SetInput(image)
    writer.Update()


# Guardado de imágenes resultado
ct_registered_path = os.path.join(RESULTS_DIR, "CT_2_registered.nrrd")
ct_diff_path = os.path.join(RESULTS_DIR, "CT_diff.nrrd")
write_image(ct2_registered, ct_registered_path)
write_image(ct_diff, ct_diff_path)

# Guardado de las transformaciones (affine y BSpline por separado)
ct_affine_path = os.path.join(TRANSFORMS_DIR, "ct_affine_transform.tfm")
ct_bspline_path = os.path.join(TRANSFORMS_DIR, "ct_bspline_transform.tfm")

tx_writer = itk.TransformFileWriterTemplate[itk.D].New()
tx_writer.SetInput(affine_transform)
tx_writer.SetFileName(ct_affine_path)
tx_writer.Update()

tx_writer = itk.TransformFileWriterTemplate[itk.D].New()
tx_writer.SetInput(bspline_transform)
tx_writer.SetFileName(ct_bspline_path)
tx_writer.Update()

print("Archivos guardados:")
for p in [ct_registered_path, ct_diff_path, ct_affine_path, ct_bspline_path]:
    print(f"  - {p}")

tiempo_etapa = time.time() - tiempo_inicio_etapa
print(f"[Etapa 4] Resampling CT y guardado — Tiempo: {tiempo_etapa:.1f}s ({tiempo_etapa/60:.1f} min)")

# Tiempo total del pipeline CT
tiempo_total = time.time() - tiempo_inicio_total
print(f"\n=== Pipeline CT completado ===")
print(f"Tiempo total: {tiempo_total:.1f}s ({tiempo_total/60:.1f} min)")

## Paso 8: Registro PET — Transferencia de transformación
Descripción breve: se aplica la misma transformación CT al par PET sin reoptimizar.

In [ ]:
# Lectura de las transformaciones CT guardadas en disco
def read_transform(path):
    # Lee una transformación ITK desde un .tfm (devuelve la primera)
    transforms = itk.transformread(path)
    if not transforms:
        raise RuntimeError("El archivo no contiene transformaciones.")
    return transforms[0]


# Marca de tiempo del pipeline PET completo
tiempo_inicio_pet = time.time()

# Etapa 1 - Lectura de transforms
tiempo_inicio_etapa = time.time()
affine_tx = read_transform(ct_affine_path)
bspline_tx = read_transform(ct_bspline_path)
tiempo_etapa = time.time() - tiempo_inicio_etapa
print(f"  Affine  : {type(affine_tx).__name__}")
print(f"  BSpline : {type(bspline_tx).__name__}")
print(f"[Etapa 1] Lectura de transforms — Tiempo: {tiempo_etapa:.1f}s ({tiempo_etapa/60:.1f} min)")

# Etapa 2 - Resampling PET
tiempo_inicio_etapa = time.time()

# CompositeTransform en el mismo orden que el registro CT (affine -> bspline)
pet_composite = itk.CompositeTransform[itk.D, Dimension].New()
pet_composite.AddTransform(affine_tx)
pet_composite.AddTransform(bspline_tx)

# Resampling de PET_2 usando PET_1 como rejilla de referencia
resampler = itk.ResampleImageFilter[ImageType, ImageType].New()
resampler.SetInput(pet2)             # PET_2
resampler.SetTransform(pet_composite)
resampler.SetReferenceImage(pet1)    # PET_1
resampler.UseReferenceImageOn()
resampler.SetDefaultPixelValue(0.0)  # SUV mínimo apropiado para PET
resampler.SetInterpolator(itk.LinearInterpolateImageFunction[ImageType, itk.D].New())
resampler.Update()
pet2_registered = resampler.GetOutput()

# Imagen diferencia |PET_1 - PET_2_registered|
sub = itk.SubtractImageFilter[ImageType, ImageType, ImageType].New()
sub.SetInput1(pet1)
sub.SetInput2(pet2_registered)
abs_filter = itk.AbsImageFilter[ImageType, ImageType].New()
abs_filter.SetInput(sub.GetOutput())
abs_filter.Update()
pet_diff = abs_filter.GetOutput()
tiempo_etapa = time.time() - tiempo_inicio_etapa
print(f"[Etapa 2] Resampling PET — Tiempo: {tiempo_etapa:.1f}s ({tiempo_etapa/60:.1f} min)")

# Guardado de resultados PET
pet_registered_path = os.path.join(RESULTS_DIR, "PET_2_registered.nrrd")
pet_diff_path = os.path.join(RESULTS_DIR, "PET_diff.nrrd")
write_image(pet2_registered, pet_registered_path)
write_image(pet_diff, pet_diff_path)

print("Registro PET completado ✓")

# Tiempo total del pipeline PET
tiempo_total = time.time() - tiempo_inicio_pet
print(f"\n=== Pipeline PET completado ===")
print(f"Tiempo total: {tiempo_total:.1f}s ({tiempo_total/60:.1f} min)")

## Paso 9: Visualización de resultados
3 vistas (axial, coronal, sagital) para cada imagen resultado.

In [ ]:
# Visualización: 6 imágenes (filas) x 3 vistas ortogonales (columnas).
# itk.array_from_image devuelve el arreglo en orden (z, y, x).
imagenes = [
    ("CT_1", ct1, "gray"),
    ("CT_2_registered", ct2_registered, "gray"),
    ("CT_diff", ct_diff, "gray"),
    ("PET_1", pet1, "hot"),
    ("PET_2_registered", pet2_registered, "hot"),
    ("PET_diff", pet_diff, "hot"),
]
vistas = ["Axial", "Coronal", "Sagital"]

fig, axes = plt.subplots(len(imagenes), len(vistas), figsize=(12, 22))

for fila, (nombre, imagen, cmap) in enumerate(imagenes):
    arr = itk.array_from_image(imagen)  # (z, y, x)
    nz, ny, nx = arr.shape
    cortes = [
        arr[nz // 2, :, :],  # Axial   (slice central en z)
        arr[:, ny // 2, :],  # Coronal (slice central en y)
        arr[:, :, nx // 2],  # Sagital (slice central en x)
    ]
    for col, (vista, corte) in enumerate(zip(vistas, cortes)):
        ax = axes[fila, col]
        ax.imshow(corte, cmap=cmap, origin="lower", aspect="auto")
        ax.set_title(f"{nombre} - {vista}", fontsize=9)
        ax.axis("off")

fig.tight_layout()
vis_path = "/content/results/visualizacion_completa.png"
fig.savefig(vis_path, dpi=150, bbox_inches="tight")
print("Visualización guardada ✓")

## Paso 10: Descarga de resultados
Descargar los archivos generados para incluirlos en el reporte.

In [ ]:
# Descarga de los resultados generados al equipo local
from google.colab import files

files.download("/content/results/visualizacion_completa.png")
files.download("/content/results/CT_2_registered.nrrd")
files.download("/content/results/PET_2_registered.nrrd")